# 4. QR Model Validation

This notebook is intentionally **load-only**. All heavy calibration, FTQR diagnostics, and SAQR table construction must be precomputed remotely and copied back into `data/processed/remote_results/qr_model_validation/`.

Expected artifact set:

- `calibration_summary.json`
- `common_intensity.parquet`
- `ftqr_intensity.parquet`
- `qr_intensities.parquet`
- `saqr_aes.parquet`
- `ftqr_small_n_table.csv`
- `ftqr_full_consumption_probabilities.csv`
- `ftqr_conditional_size_distributions.csv`
- `qr_qru_ftqr_simulation_summary.csv`

If any required file is missing, the notebook stops immediately with a clear message.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.colors import LinearSegmentedColormap

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RESULTS_DIR = ROOT / "data/processed/remote_results/qr_model_validation"
AES_CROP_MAX = 30

def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing precomputed file {path}. Run the remote precompute script on gpu.enst.fr.")
    return path

summary = json.loads(require_file(RESULTS_DIR / "calibration_summary.json").read_text())
common_intensity = pd.read_parquet(require_file(RESULTS_DIR / "common_intensity.parquet"))
ftqr_intensity = pd.read_parquet(require_file(RESULTS_DIR / "ftqr_intensity.parquet"))
qr_intensities = pd.read_parquet(require_file(RESULTS_DIR / "qr_intensities.parquet"))
saqr_aes = pd.read_parquet(require_file(RESULTS_DIR / "saqr_aes.parquet"))
ftqr_small = pd.read_csv(require_file(RESULTS_DIR / "ftqr_small_n_table.csv"))
ftqr_full_probs = pd.read_csv(require_file(RESULTS_DIR / "ftqr_full_consumption_probabilities.csv"))
ftqr_cond_sizes = pd.read_csv(require_file(RESULTS_DIR / "ftqr_conditional_size_distributions.csv"))
ftqr_sim = pd.read_csv(require_file(RESULTS_DIR / "qr_qru_ftqr_simulation_summary.csv"))

summary_df = pd.DataFrame([summary])
display(summary_df)


## FTQR Decomposition

The FTQR section verifies that the implementation correctly decomposes QR cancel and market intensity into partial and full-consumption components.

In [ ]:
ftqr_identity_m = (ftqr_small["lambda_M_qr"] - (ftqr_small["lambda_M_ftqr"] + ftqr_small["lambda_M_all_ftqr"])).abs().max()
ftqr_identity_c = (ftqr_small["lambda_C_qr"] - (ftqr_small["lambda_C_ftqr"] + ftqr_small["lambda_C_all_ftqr"])).abs().max()
display(pd.DataFrame({"metric": ["max |M identity error|", "max |C identity error|"], "value": [ftqr_identity_m, ftqr_identity_c]}))

plot_df = ftqr_small[ftqr_small["n"] <= 40].copy()
marker_kws = dict(marker="o", markersize=4)
fig, axes = plt.subplots(1, 2, figsize=(16, 4), sharex=True)
axes[0].plot(plot_df["n"], plot_df["lambda_M_qr"], linewidth=2, label="QR lambda_M", **marker_kws)
axes[0].plot(plot_df["n"], plot_df["lambda_M_ftqr"], linewidth=2, label="FTQR lambda_M_partial", **marker_kws)
axes[0].plot(plot_df["n"], plot_df["lambda_M_all_ftqr"], linewidth=2, label="FTQR lambda_M_all", **marker_kws)
axes[0].plot(plot_df["n"], plot_df["lambda_M_ftqr"] + plot_df["lambda_M_all_ftqr"], linewidth=2, linestyle="--", label="FTQR total", **marker_kws)
axes[0].set_title("Market Intensity: QR vs FTQR")
axes[0].set_xlabel("Queue size n (AES)")
axes[0].set_ylabel("Intensity")
axes[0].set_xlim(1, 40)
axes[0].grid(alpha=0.2)
axes[0].legend()

axes[1].plot(plot_df["n"], plot_df["lambda_C_qr"], linewidth=2, label="QR lambda_C", **marker_kws)
axes[1].plot(plot_df["n"], plot_df["lambda_C_ftqr"], linewidth=2, label="FTQR lambda_C_partial", **marker_kws)
axes[1].plot(plot_df["n"], plot_df["lambda_C_all_ftqr"], linewidth=2, label="FTQR lambda_C_all", **marker_kws)
axes[1].plot(plot_df["n"], plot_df["lambda_C_ftqr"] + plot_df["lambda_C_all_ftqr"], linewidth=2, linestyle="--", label="FTQR total", **marker_kws)
axes[1].set_title("Cancel Intensity: QR vs FTQR")
axes[1].set_xlabel("Queue size n (AES)")
axes[1].set_xlim(1, 40)
axes[1].grid(alpha=0.2)
axes[1].legend()
fig.tight_layout()

prob_plot = ftqr_full_probs[ftqr_full_probs["n"] <= 25].copy()
fig, ax = plt.subplots(figsize=(8, 4))
for eta, label in [("M", "P(s >= q | M, n)"), ("C", "P(s >= q | C, n)")]:
    sub = prob_plot[prob_plot["eta"] == eta]
    ax.plot(sub["n"], sub["p_full"], linewidth=2, marker="o", markersize=4, label=label)
ax.set_xlabel("Queue size n (AES)")
ax.set_ylabel("Probability")
ax.set_title("Full-consumption probabilities")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()

display(ftqr_sim)
display(Markdown(
    """
### FTQR Interpretation

- FTQR is implemented correctly when the decomposition identities hold numerically.
- FTQR redistributes intensity between partial and full-consumption events.
- FTQR does not alter the total `lambda_M(n)` or `lambda_C(n)` shape by itself; it only decomposes that shape.
- On this Bund dataset, full-consumption effects are concentrated mostly at the very smallest queues.
"""
))


## Conditional vs Stationary Size Distributions

The plots below compare the unconditional order-size distribution with the conditional distributions for selected small queue sizes.

In [ ]:
for eta in ["M", "C"]:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
    eta_df = ftqr_cond_sizes[ftqr_cond_sizes["eta"] == eta].copy()
    for ax, n in zip(axes, [3, 5, 10]):
        sub = eta_df[eta_df["n"] == n].copy()
        if sub.empty:
            continue
        sub["prob_cond"] = sub["count"] / sub["count"].sum()
        sub["prob_uncond"] = sub["count_uncond"] / sub["count_uncond"].sum()
        sub = sub.sort_values("size").head(20)
        x = np.arange(len(sub))
        ax.bar(x - 0.2, sub["prob_uncond"], width=0.4, label="P(s)")
        ax.bar(x + 0.2, sub["prob_cond"], width=0.4, label=f"P(s | q={n})")
        ax.set_title(f"{eta}, n={n}")
        ax.set_xlabel("Order-size bins")
        ax.set_xticks(x)
        ax.set_xticklabels(sub["size"].astype(int), rotation=90)
        ax.grid(axis="y", alpha=0.2)
    axes[0].set_ylabel("Probability")
    axes[0].legend()
    fig.tight_layout()


## QR vs QRU Intensities

QR and QRU share the same calibrated intensity curves; the difference between them is in the simulated event-size mechanism, not in the calibration itself.

In [ ]:
qr_plot = common_intensity[common_intensity["n"] <= 40].copy()
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=True)
for ax, col, title in zip(axes, ["lambda_L", "lambda_C", "lambda_M"], ["lambda_L", "lambda_C", "lambda_M"]):
    ax.plot(qr_plot["n"], qr_plot[col], linewidth=2, marker="o", markersize=4, label="QR")
    ax.plot(qr_plot["n"], qr_plot[col], linewidth=2, linestyle="--", marker="x", markersize=4, label="QRU")
    ax.set_title(title)
    ax.set_xlabel("Queue size n (AES)")
    ax.set_xlim(1, 40)
    ax.grid(alpha=0.2)
    ax.legend()
axes[0].set_ylabel("Intensity")
fig.tight_layout()


## SAQR Diagnostics

The SAQR section uses AES-normalized queue size and AES-normalized order size. Support heatmaps show observation counts; intensity heatmaps show `lambda_(eta,s)(n)`; row-normalized heatmaps show the conditional size composition within each queue-size row.

In [ ]:
paper_cmap = LinearSegmentedColormap.from_list(
    "paper_like",
    ["#000000", "#08306b", "#2171b5", "#6baed6", "#fff7bc"],
)

support_by_n = saqr_aes.groupby(["eta", "n"], as_index=False)["count"].sum()
support_by_s = saqr_aes.groupby(["eta", "size_aes"], as_index=False)["count"].sum()
display(pd.DataFrame({
    "metric": ["queue n min", "queue n max", "size_aes min", "size_aes max", "rows"],
    "value": [saqr_aes["n"].min(), saqr_aes["n"].max(), saqr_aes["size_aes"].min(), saqr_aes["size_aes"].max(), len(saqr_aes)],
} ))
display(support_by_n[support_by_n["n"] <= 20].head(20))
display(support_by_s[support_by_s["size_aes"] <= 20].head(20))

def build_heatmap(df, eta, value_col, triangular):
    sub = df[(df["eta"] == eta) & (df["n"] <= AES_CROP_MAX) & (df["size_aes"] <= AES_CROP_MAX)].copy()
    heat = sub.pivot(index="n", columns="size_aes", values=value_col).sort_index().sort_index(axis=1)
    if triangular:
        for n in heat.index:
            valid = heat.columns <= n
            heat.loc[n, ~valid] = np.nan
    return heat

support_heatmaps = {eta: build_heatmap(saqr_aes, eta, "count", triangular=(eta != "L")) for eta in ["L", "C", "M"]}
lambda_heatmaps = {eta: build_heatmap(saqr_aes, eta, "lambda_eta_size", triangular=(eta != "L")) for eta in ["L", "C", "M"]}
row_heatmaps = {eta: build_heatmap(saqr_aes, eta, "row_prob", triangular=(eta != "L")) for eta in ["L", "C", "M"]}

fig, axes = plt.subplots(3, 3, figsize=(16, 14))
for row_idx, eta in enumerate(["L", "C", "M"]):
    for col_idx, (title, heatmaps) in enumerate([("Support", support_heatmaps), ("Intensity", lambda_heatmaps), ("Row-normalized", row_heatmaps)]):
        ax = axes[row_idx, col_idx]
        heat = heatmaps[eta]
        im = ax.imshow(heat.to_numpy(), origin="lower", aspect="auto", cmap=paper_cmap)
        ax.set_title(f"{eta}: {title}")
        ax.set_xlabel("Order size (AES)")
        ax.set_ylabel("Queue size (AES)")
        ax.set_xticks(range(len(heat.columns)))
        ax.set_xticklabels(heat.columns.astype(int), rotation=90)
        ax.set_yticks(range(len(heat.index)))
        ax.set_yticklabels(heat.index.astype(int))
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()

diag_rows = []
for eta in ["C", "M"]:
    sub = saqr_aes[(saqr_aes["eta"] == eta) & (saqr_aes["n"] <= AES_CROP_MAX) & (saqr_aes["size_aes"] <= AES_CROP_MAX)].copy()
    total = sub["lambda_eta_size"].sum()
    diag = sub[np.abs(sub["size_aes"] - sub["n"]) <= 1]["lambda_eta_size"].sum()
    above = sub[sub["size_aes"] > sub["n"]]["lambda_eta_size"].sum()
    below = sub[sub["size_aes"] < sub["n"]]["lambda_eta_size"].sum()
    diag_rows.append({
        "eta": eta,
        "diag_mass": diag / total if total else np.nan,
        "above_diag_mass": above / total if total else np.nan,
        "below_diag_mass": below / total if total else np.nan,
    })
diag_df = pd.DataFrame(diag_rows)
display(diag_df.style.format({"diag_mass": "{:.4f}", "above_diag_mass": "{:.4f}", "below_diag_mass": "{:.4f}"}))

display(Markdown(
    """
### SAQR Interpretation

- Both axes are AES-normalized: queue size on the y-axis and order size on the x-axis.
- The upper-triangular `s > q` region is masked for cancels and market orders because it is not meaningful in the queue-consumption interpretation.
- If cancel and market heatmaps still do not show a strong near-diagonal structure after these corrections, the remaining gap is empirical rather than a notebook presentation bug.
"""
))
